# ⚡ 03 — OTA Firmware Update

Push a new firmware binary to one or more bots over HTTP.

Workflow:
1. Discover online fleet
2. Probe current firmware versions via `/api/info`
3. Select target bot(s)
4. Upload firmware binary via `POST /api/ota`
5. Poll for reboot / confirm new version

> **Caution:** Only push firmware that targets the correct hardware platform.  
> Flashing the wrong binary can brick a bot until manual recovery.

---

## Setup

In [ ]:
import sys, time
from pathlib import Path

REPO_ROOT = Path().resolve().parent
FLEET_MGR = REPO_ROOT / 'fleet-manager'
if str(FLEET_MGR) not in sys.path:
    sys.path.insert(0, str(FLEET_MGR))

from fleet_manager import scan, probe_fleet, ota_update, ota_update_fleet

FLEET_YAML = REPO_ROOT / 'docs' / 'fleet.yaml'
print('fleet_manager imported ✓')

---
## 1 · Discover Fleet

In [ ]:
BOT_FAMILIES = ['rfbot', 'mybot', 'carbot', 'paulbot', 'simplebot', 'dogbot']

print('Scanning…')
fleet = scan(BOT_FAMILIES, workers=20, timeout=1.0, yaml_path=FLEET_YAML)
print(fleet.summary())

for bot in fleet.online:
    print(f'  ● {bot.hostname:<22}  {bot.ip}')

---
## 2 · Probe Current Firmware Versions

In [ ]:
if fleet.online:
    print(f'Probing {len(fleet.online)} online bot(s) for firmware info…')
    probe_fleet(fleet, timeout=2.0)

    print(f'\n{"Hostname":<24} {"Platform":<14} {"Firmware":<12} {"Uptime":>10}')
    print('─' * 64)
    for bot in fleet.online:
        uptime = f'{bot.uptime_s}s' if bot.uptime_s >= 0 else '—'
        fw     = bot.fw_version or '(unknown)'
        plat   = bot.platform or '(unknown)'
        print(f'  {bot.hostname:<22} {plat:<14} {fw:<12} {uptime:>10}')
else:
    print('No online bots to probe.')

---
## 3 · Configure Update Target

In [ ]:
# ── Set these before running ──────────────────────────────────────────────────

# Path to the .bin firmware file to push
FIRMWARE_PATH = Path(REPO_ROOT / 'firmware' / 'build' / 'firmware.bin')

# Strategy:
#   'single'   — push to one specific bot (set TARGET_HOSTNAME below)
#   'platform' — push to all online bots of a given platform
#   'all'      — push to every online bot (use with caution!)
STRATEGY = 'single'

TARGET_HOSTNAME = 'paulbot0.local'  # used when STRATEGY == 'single'
TARGET_PLATFORM = 'dogbot_v1'       # used when STRATEGY == 'platform'

# ── Resolve targets ───────────────────────────────────────────────────────────
if STRATEGY == 'single':
    target_bots = [b for b in fleet.online if b.hostname == TARGET_HOSTNAME]
elif STRATEGY == 'platform':
    target_bots = fleet.by_platform(TARGET_PLATFORM)
else:  # 'all'
    target_bots = fleet.online

print(f'Strategy  : {STRATEGY}')
print(f'Firmware  : {FIRMWARE_PATH}  (exists={FIRMWARE_PATH.exists()})')
print(f'Targets   : {len(target_bots)} bot(s)')
for b in target_bots:
    print(f'  → {b.hostname}  ({b.ip})')

---
## 4 · Push OTA Update

In [ ]:
if not target_bots:
    print('⚠️  No target bots. Adjust STRATEGY or TARGET_HOSTNAME above.')
elif not FIRMWARE_PATH.exists():
    print(f'⚠️  Firmware not found: {FIRMWARE_PATH}')
    print('    Build the firmware first: idf.py build')
else:
    print(f'Uploading {FIRMWARE_PATH.name} ({FIRMWARE_PATH.stat().st_size / 1024:.1f} KB)…\n')

    results: dict[str, bool] = {}

    for bot in target_bots:
        print(f'  → {bot.hostname:<22} ', end='', flush=True)
        t0 = time.perf_counter()
        ok = ota_update(bot, FIRMWARE_PATH, timeout=60.0)
        elapsed = time.perf_counter() - t0
        results[bot.hostname] = ok
        print(f'{"OK" if ok else "FAILED"}  ({elapsed:.1f}s)')

    success = sum(1 for v in results.values() if v)
    print(f'\nDone: {success}/{len(target_bots)} bots updated successfully.')

---
## 5 · Verify New Firmware (after bot reboots)

In [ ]:
REBOOT_WAIT_S = 15   # seconds to wait for the bot to come back online

if not target_bots:
    print('Nothing to verify.')
else:
    print(f'Waiting {REBOOT_WAIT_S}s for bots to reboot…')
    for remaining in range(REBOOT_WAIT_S, 0, -1):
        print(f'\r  {remaining}s remaining…  ', end='', flush=True)
        time.sleep(1)
    print('\r  Done waiting.           ')

    # Re-scan just the target bots
    target_hostnames = {b.hostname for b in target_bots}
    target_bases = list({h.split('.')[0].rstrip('0123456789') for h in target_hostnames})

    print('Re-scanning target bots…')
    new_fleet = scan(target_bases, workers=10, timeout=2.0, yaml_path=FLEET_YAML)
    probe_fleet(new_fleet, timeout=3.0)

    print(f'\n{"Hostname":<24} {"Status":<10} {"Old FW":<12} {"New FW":<12}')
    print('─' * 60)

    old_versions = {b.hostname: b.fw_version for b in target_bots}
    for bot in new_fleet.bots:
        if bot.hostname in target_hostnames:
            old_fw = old_versions.get(bot.hostname, '—')
            new_fw = bot.fw_version or '(unknown)'
            changed = '✓ updated' if new_fw != old_fw else '— same'
            print(f'  {bot.hostname:<22} {bot.status:<10} {old_fw:<12} {new_fw:<12}  {changed}')

---
## 6 · Fleet Update Progress Chart

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    # Show old vs new firmware version across all online bots
    bots  = [b for b in fleet.online if b.fw_version]
    names = [b.hostname.replace('.local', '') for b in bots]
    fws   = [b.fw_version for b in bots]

    if bots:
        unique_fw = sorted(set(fws))
        cmap = plt.colormaps['cool']
        color_map = {fw: cmap(i / max(1, len(unique_fw) - 1)) for i, fw in enumerate(unique_fw)}
        colors = [color_map[fw] for fw in fws]

        fig, ax = plt.subplots(figsize=(max(6, len(bots) * 1.2), 4))
        fig.patch.set_facecolor('#0f172a')
        ax.set_facecolor('#1e293b')

        bars = ax.bar(names, [1] * len(bots), color=colors, edgecolor='#0f172a', linewidth=1.5)

        for bar, fw in zip(bars, fws):
            ax.text(bar.get_x() + bar.get_width() / 2, 0.5, fw,
                    ha='center', va='center', fontsize=10,
                    color='white', fontweight='bold')

        patches = [mpatches.Patch(color=color_map[fw], label=fw) for fw in unique_fw]
        ax.legend(handles=patches, title='Firmware', loc='upper right',
                  facecolor='#1e293b', edgecolor='#334155', labelcolor='#94a3b8',
                  title_fontsize=10)

        ax.set_yticks([])
        ax.set_title('Firmware Versions Across Fleet', color='white', fontsize=13)
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right', color='#94a3b8', fontsize=9)
        for sp in ax.spines.values():
            sp.set_color('#334155')

        plt.tight_layout()
        plt.show()
    else:
        print('No firmware version data available (probe_fleet was not run or bots returned no info).')

except ImportError:
    print('matplotlib not installed — run: pip install matplotlib')